# 15 SMDPL from CosmoSim

Data for a second cross-simulation test: the SMDPL N-body run (400 h⁻¹ Mpc, 3840³ particles, Planck cosmology).

1. **Density grid.** CosmoSim's `smdpl.dens512_z0` table (z = 0 overdensity on a 512³ grid, 0.78 h⁻¹ Mpc cells) through its TAP service, in 27 chunks of at most 19 x-slices, because a query may return at most 5,000,000 rows. Each chunk is saved to Drive as Parquet; chunks already there are skipped, so the cell can be rerun after an interruption.
2. **Check and assemble** the chunks into one 512³ array: every (ix, iy, iz) exactly once.

No GPU needed.

**Token.** The TAP service needs your CosmoSim API token (user menu, top right, "API Token"). Store it as a Colab secret named `COSMOSIM_TOKEN` (key icon in the left sidebar) and give this notebook access. It is never written into the notebook.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install pyvo

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyvo
import requests

TAP_URL = 'https://www.cosmosim.org/tap'
QUEUE = '1h'                 # the web form's queues: 1m, 1h, 2h, 5h
OUT = Path('/content/drive/MyDrive/data/SMDPL/dens512')
OUT.mkdir(parents=True, exist_ok=True)
N, STEP = 512, 19            # 19 x-slices x 512^2 = 4,980,736 rows < 5,000,000

session = requests.Session()
session.headers['Authorization'] = f"Token {userdata.get('COSMOSIM_TOKEN')}"
tap = pyvo.dal.TAPService(TAP_URL, session=session)

# The languages the service accepts, as TAP names them ("name-version"); use the PostgreSQL one.
langs = []
for lang in tap.get_tap_capability().languages:
    versions = [v.content for v in lang.versions if v.content] if lang.versions else []
    langs += [f'{lang.name}-{v}' for v in versions] or [lang.name]
print('languages:', langs)
LANGUAGE = next((l for l in langs if l.lower().startswith('postgres')), None)
assert LANGUAGE, 'no PostgreSQL language listed; paste the list above'
print('using', LANGUAGE)

# A tiny synchronous query to check the token, the language and the table.
test = tap.run_sync('SELECT ix, iy, iz, dens FROM smdpl.dens512_z0 WHERE ix = 0 AND iy = 0 AND iz < 3',
                    language=LANGUAGE).to_table()
print(test)

## 1. Download the chunks

One asynchronous job at a time: submit, wait, download, save, then the next. Expect several minutes per chunk. If the session drops, rerun this cell; finished chunks are skipped.

In [ ]:
chunks = [(a, min(a + STEP - 1, N - 1)) for a in range(0, N, STEP)]
assert len(chunks) == 27 and chunks[-1] == (494, 511)

for a, b in chunks:
    path = OUT / f'dens512_ix{a:03d}_{b:03d}.parquet'
    if path.exists():
        print(path.name, 'already there')
        continue
    query = f'SELECT ix, iy, iz, dens FROM smdpl.dens512_z0 WHERE ix BETWEEN {a} AND {b}'
    t0 = time.time()
    job = tap.submit_job(query, language=LANGUAGE, queue=QUEUE, runid=f'dens512_ix{a:03d}_{b:03d}')
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR', 'ABORTED'], timeout=3600)
    if job.phase != 'COMPLETED':
        job.raise_if_error()
        raise RuntimeError(f'{path.name}: job ended as {job.phase}')
    df = job.fetch_result().to_table().to_pandas()
    expected = (b - a + 1) * N * N
    if len(df) != expected:
        raise RuntimeError(f'{path.name}: {len(df)} rows, expected {expected}')
    df = df.astype({'ix': 'int16', 'iy': 'int16', 'iz': 'int16', 'dens': 'float32'})
    df.to_parquet(path, index=False)
    job.delete()                      # frees the job's table in your CosmoSim space
    print(f'{path.name}: {len(df):,} rows in {time.time() - t0:.0f} s', flush=True)

## 2. Check and assemble

Every cell must appear exactly once. Saves the grid as `MyDrive/data/SMDPL/smdpl_dens512_z0.npy` (float32, indexed [ix, iy, iz]), plus a few numbers to sanity-check: an overdensity field averages 0 and cannot go below -1.

In [ ]:
grid = np.full((N, N, N), np.nan, dtype=np.float32)
seen = np.zeros((N, N, N), dtype=np.uint8)
for path in sorted(OUT.glob('dens512_ix*.parquet')):
    df = pd.read_parquet(path)
    ix, iy, iz = (df[c].to_numpy() for c in ('ix', 'iy', 'iz'))
    grid[ix, iy, iz] = df['dens'].to_numpy()
    np.add.at(seen, (ix, iy, iz), 1)
    del df
print('cells seen once:', int((seen == 1).sum()), 'of', N ** 3)
assert (seen == 1).all(), f'missing: {(seen == 0).sum()}, repeated: {(seen > 1).sum()}'
print(f'mean {grid.mean():.4f}, min {grid.min():.4f}, max {grid.max():.1f}')
np.save(OUT.parent / 'smdpl_dens512_z0.npy', grid)
print('saved', OUT.parent / 'smdpl_dens512_z0.npy')